In [1]:
#앙상블 학습

#데이터 준비 및 전처리
import numpy as np
import pandas as pd
import requests
from io import StringIO
url = r'https://bit.ly/wine_csv_data'
try: 
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    wine = pd.read_csv(StringIO(r.text), low_memory=False)
except ConnectionError:
    pass

if wine is not False:
    wine.head()
    print(wine.columns)
    print(wine.info())
    print(wine.describe())
    print(wine['class'].unique())

from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(wine[['alcohol', 'sugar', 'pH']], wine['class'], test_size=0.2, random_state=42, stratify=wine['class'])
print(train_input.shape, train_target.shape)

Index(['alcohol', 'sugar', 'pH', 'class'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   alcohol  6497 non-null   float64
 1   sugar    6497 non-null   float64
 2   pH       6497 non-null   float64
 3   class    6497 non-null   float64
dtypes: float64(4)
memory usage: 203.2 KB
None
           alcohol        sugar           pH        class
count  6497.000000  6497.000000  6497.000000  6497.000000
mean     10.491801     5.443235     3.218501     0.753886
std       1.192712     4.757804     0.160787     0.430779
min       8.000000     0.600000     2.720000     0.000000
25%       9.500000     1.800000     3.110000     1.000000
50%      10.300000     3.000000     3.210000     1.000000
75%      11.300000     8.100000     3.320000     1.000000
max      14.900000    65.800000     4.010000     1.000000
[0. 1.]
(5197, 3) (5197,)


In [2]:
#앙상블 분석 - 랜덤포레스트 모델
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=1, random_state=42)
scores = cross_validate(rf, train_input, train_target, n_jobs=1, return_train_score=True)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

#랜덤포레스트의 특성 중요도: 특성을 exp(특성 수)만큼 매 노드마다 랜덤하게 추출, 특성의 랜덤한 사용으로 인한 특정 특성 사용 방지, 상대적으로 고른 기회 -> 과적합 방지, 일반화 성능 높음
rf.fit(train_input, train_target)
print(rf.feature_importances_)

#OOB 샘플(Out Of Bag) -> 부트스트랩 후 남은 데이터를 검증 데이터로 활용
rf = RandomForestClassifier(oob_score=True, random_state=42, n_jobs=1)
rf.fit(train_input, train_target)
print(rf.oob_score_)


0.9981720130385032
0.8937817428000298
[0.2301315  0.50081183 0.26905668]
0.8964787377333077


In [3]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(oob_score=True, n_jobs=1, random_state=42)
from sklearn.model_selection import cross_validate, StratifiedKFold
cv = StratifiedKFold(shuffle=True, random_state=42)
scores = cross_validate(rf, train_input, train_target, cv=cv, return_train_score=True, n_jobs=1)
print(np.mean(scores['test_score']))
print(np.mean(scores['train_score']))

rf.fit(train_input, train_target)
print(rf.feature_importances_)
print(rf.score(train_input, train_target))
print(rf.oob_score_)


0.8885885096616569
0.9980758013714472
[0.2301315  0.50081183 0.26905668]
0.9978833942659227
0.8964787377333077


In [10]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=1, random_state=42)
scores = cross_validate(et, train_input, train_target, cv=cv, return_train_score=True)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))
et.fit(train_input, train_target)
print(et.feature_names_in_, et.feature_importances_)

0.9981720130385032
0.889165617827793
['alcohol' 'sugar' 'pH'] [0.20321321 0.51571223 0.28107456]


In [16]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
scores = cross_validate(gb, train_input, train_target, cv=cv, n_jobs=1, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42, subsample=0.5)
scores = cross_validate(gb, train_input, train_target, cv=cv, return_train_score=True, n_jobs=1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))
gb.fit(train_input, train_target)
print(gb.feature_names_in_, gb.feature_importances_)

0.8866654563551364 0.8743510772192197
0.9391956727775828 0.865880284297031
['alcohol' 'sugar' 'pH'] [0.20422158 0.47261751 0.32316091]


In [27]:
from sklearn.ensemble import HistGradientBoostingClassifier
hg = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hg, train_input, train_target, cv=cv, return_train_score=True)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

#특성 중요도 계산 - 특성 중요도는 왜 필요하지? 어떤 특성에 관심을 가져야 하는가?
from sklearn.inspection import permutation_importance
hg.fit(train_input, train_target)
imp1 = permutation_importance(hg, train_input, train_target, n_repeats=10, random_state=42, n_jobs=1)
print(hg.feature_names_in_)
print(imp1['importances_mean'])

imp = permutation_importance(hg, test_input, test_target, n_repeats=10, random_state=42)
print(imp['importances_mean'])
print()

print(hg.score(test_input, test_target))

0.9330860062878346
0.8787756348560005
['alcohol' 'sugar' 'pH']
[0.09374639 0.23954204 0.08664614]
[0.04953846 0.20176923 0.04776923]

0.8776923076923077


[np.float64(0.166),
 np.float64(0.675),
 np.float64(0.16),
 '/',
 np.float64(0.223),
 np.float64(0.57),
 np.float64(0.206)]